<a href="https://colab.research.google.com/github/Marshud/ml-classes-01/blob/main/JAX_First_Look.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# (J)IT Compilation (A)utomatic Differentiation (X)LA Accelerated Linear Algebra

### JAX as NumPy

In [1]:
import jax
import jax.numpy as jnp

In [2]:
a = jnp.array([1, 2, 3])
b = jnp.array([4, 5, 6])

print(a+b)
print(jnp.sqrt(a))
print(jnp.mean(a))
print(a.reshape(-1, 1))

# a[1] = 10 // You can't do this in JAX but passes in numpy

[5 7 9]
[1.        1.4142135 1.7320508]
2.0
[[1]
 [2]
 [3]]


### JIT Compilation

In [3]:
import time

@jax.jit # This is ueful for training loops that are executed multiple times
def myfunction(x):
  return jnp.where(x % 2 == 0, x / 2, 3 * x + 1) # Collatz Sequence

arr = jnp.arange(10)

_ = myfunction(arr) # JIT compilation

start = time.perf_counter()
myfunction(arr).block_until_ready() # The block_until_ready is added because JAX hands over the execution back
end = time.perf_counter()

print(end-start)

print(jax.make_jaxpr(myfunction)(arr))

# It doesn't allow compilation of all functions. If a function has a condition in which the function argument is used, it doesn't compile
# def f(x):
#   if x % 2 == 0:
#     return 1
#   else:
#     return 0



0.0002898239999922225
{ lambda ; a:i32[10]. let
    b:f32[10] = jit[
      name=myfunction
      jaxpr={ lambda ; a:i32[10]. let
          c:i32[10] = jit[
            name=remainder
            jaxpr={ lambda ; a:i32[10] d:i32[]. let
                e:i32[] = convert_element_type[new_dtype=int32 weak_type=False] d
                f:bool[] = eq e 0:i32[]
                g:i32[] = jit[
                  name=_where
                  jaxpr={ lambda ; f:bool[] h:i32[] e:i32[]. let
                      g:i32[] = select_n f e h
                    in (g,) }
                ] f 1:i32[] e
                i:i32[10] = rem a g
                j:bool[10] = ne i 0:i32[]
                k:bool[10] = lt i 0:i32[]
                l:bool[] = lt g 0:i32[]
                m:bool[10] = ne k l
                n:bool[10] = and m j
                o:i32[10] = add i g
                c:i32[10] = select_n n i o
              in (c,) }
          ] a 2:i32[]
          p:bool[10] = eq c 0:i32[]
          q:f32[

### Automatic Differentiation

In [4]:
def square(x):
  return x ** 2

# x = 10
# f(x) = x ^ 2 = 100
# f'(x) = 2x = 20
# f''(x) = 2 = 2
# f'''(x) = 0 = 0

value = 10.0 # Differentiation works well with floats
print(square(value))
print(jax.grad(square)(value))
print(jax.grad(jax.grad(square))(value))
print(jax.grad(jax.grad(jax.grad(square)))(value))

100.0
20.0
2.0
0.0


In [5]:
# Partial derivatives are also possible
def f( x, y, z):
  return x ** 2 + 2 * y ** 2 + 3 * z ** 2

x, y, z = 2.0, 2.0, 2.0

# x^2 + 2y^2 + 3z^2
# df/dx = 2x = 4
# df/dy = 4y = 8
# df/dz = 6z = 12

print(f(x, y, z))
print(jax.grad(f, argnums=0)(x, y, z))
print(jax.grad(f, argnums=1)(x, y, z))
print(jax.grad(f, argnums=2)(x, y, z))

24.0
4.0
8.0
12.0


In [6]:
# Passing arguments for automatic differentiation is better with Arrays
def f2(arr):
  return arr[0] ** 2 + 2 * arr[1] ** 2 + 3 * arr[2] ** 2

print(f2([x, y, z]))
print(jax.grad(f2)([x, y, z]))

24.0
[Array(4., dtype=float32, weak_type=True), Array(8., dtype=float32, weak_type=True), Array(12., dtype=float32, weak_type=True)]


### Automatic Vectorization

I have a function that works on one value, I want to apply it to a batch

In [7]:
import time
import numpy as np

key = jax.random.key(24)

W = jax.random.normal(key, (150, 100)) # 100 values per input sample, 150 neurons in next layer
X = jax.random.normal(key, (10, 100))

def calculate_ouput(x):
  return jnp.dot(W, x)

# Inefficient
def batched_calculation_loop(X):
  return jnp.stack([calculate_ouput(x) for x in X])

def batched_calculation_manual(X):
  return jnp.dot(X, W.T)

# Alternative
batched_calculation_vmap = jax.vmap(calculate_ouput)

start = time.perf_counter()
batched_calculation_loop(X)
end = time.perf_counter()
print(end-start)

start = time.perf_counter()
batched_calculation_manual(X)
end = time.perf_counter()
print(end-start)

start = time.perf_counter()
batched_calculation_vmap(X)
end = time.perf_counter()
print(end-start)

np.testing.assert_allclose(batched_calculation_loop(X), batched_calculation_manual(X), atol=1e-4, rtol=1e-4)
np.testing.assert_allclose(batched_calculation_manual(X), batched_calculation_vmap(X), atol=1e-4, rtol=1e-4)


0.6910310089997438
0.14254700299989054
0.21943984700010333


In [8]:
# Randomness
# Keys are disposable and anytime you want to use random, you provide a key and it's used once

key = jax.random.key(42)

# Splitting keys
jax.random.normal(key)
key1, key2 = jax.random.split(key)

keys = jax.random.split(key, 10)
print(keys)

Array((10,), dtype=key<fry>) overlaying:
[[1832780943  270669613]
 [  64467757 2916123636]
 [2465931498  255383827]
 [3134548294  894150801]
 [2954079971 3276725750]
 [2765691542  824333390]
 [2768684296 3055579793]
 [2547012911 1371500959]
 [1016697191 2390192106]
 [1128875147 2463678267]]


### Training a simple neural network with JAX

In [9]:
import jax
import jax.numpy as jnp

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = load_iris()
X = data.data
y = data.target.reshape(-1, 1)
print(y)

[[0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [0]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [1]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]
 [2]]


In [10]:
encoder = OneHotEncoder(sparse_output=False)
y = encoder.fit_transform(y)
print(y)

[[1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [1. 0. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0. 1. 0.]
 [0.

In [11]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.transform(X_test)

In [12]:
import jax
import jax.numpy as jnp
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler

data = load_iris()
X = data.data
y = data.target.reshape(-1, 1)

# input -> hidden layer 1 -> hidden layer 2 -> output
def init_params(input_dim, hidden_dim1, hidden_dim2, output_dim, random_key):
  # Split into 3 keys for the 3 weight matrices
  random_keys = jax.random.split(random_key, 3)

  W1 = jax.random.normal(random_keys[0], (input_dim, hidden_dim1)) * 0.1
  b1 = jnp.zeros((hidden_dim1, ))

  W2 = jax.random.normal(random_keys[1], (hidden_dim1, hidden_dim2)) * 0.1
  b2 = jnp.zeros((hidden_dim2, ))

  W3 = jax.random.normal(random_keys[2], (hidden_dim2, output_dim)) * 0.1
  b3 = jnp.zeros((output_dim, ))

  return [W1, b1, W2, b2, W3, b3]

@jax.jit
def forward(params, X):
  W1, b1, W2, b2, W3, b3 = params
  h1 = jax.nn.relu(jnp.dot(X, W1) + b1)
  h2 = jax.nn.relu(jnp.dot(h1, W2) + b2)
  logits = jnp.dot(h2, W3) + b3
  return logits

@jax.jit
def loss_fn(params, x, y, l2_reg=0.0001):
  logits = forward(params, x)
  # Use cross entropy directly for stability
  softmax_x = jax.nn.softmax(logits)
  loss = -jnp.mean(jnp.sum(y * jnp.log(softmax_x + 1e-8), axis=1))
  l2_loss = l2_reg * sum([jnp.sum(w ** 2) for w in params[::2]])
  return loss + l2_loss

@jax.jit
def train_step(params, x, y, lr):
  grads = jax.grad(loss_fn)(params, x, y)
  return [param - lr * grad for param, grad in zip(params, grads)]

@jax.jit
def accuracy(params, x, y):
  preds = jnp.argmax(forward(params, x), axis=1)
  targets = jnp.argmax(y, axis=1)
  return jnp.mean(preds == targets)

def data_loader(X, y, batch_size):
  for i in range(0, len(X), batch_size):
    yield X[i:i+batch_size], y[i:i+batch_size]

In [17]:
random_key = jax.random.key(int(time.time()))
input_dim = X_train.shape[1]
hidden_dim1 = 16
hidden_dim2 = 8
output_dim = y_train.shape[1]
learning_rate = 0.005
batch_size = 16
epochs = 200


params = init_params(input_dim, hidden_dim1, hidden_dim2, output_dim, random_key)

for epoch in range(epochs):
  for X_batch, y_batch in data_loader(X_train, y_train, batch_size):
    params = train_step(params, X_batch, y_batch, learning_rate)

    if epoch % 10 == 0:
      train_acc = accuracy(params, X_train, y_train)
      test_acc = accuracy(params, X_test, y_test)

      print(f"Epoch: {epoch}, Train Acc: {train_acc:.4f}, Test Acc: {test_acc:.4f}")

print(f"Final Test Acc: {accuracy(params, X_test, y_test)}")

Epoch: 0, Train Acc: 0.2417, Test Acc: 0.3333
Epoch: 0, Train Acc: 0.2417, Test Acc: 0.3333
Epoch: 0, Train Acc: 0.2667, Test Acc: 0.3333
Epoch: 0, Train Acc: 0.2500, Test Acc: 0.3333
Epoch: 0, Train Acc: 0.2583, Test Acc: 0.1667
Epoch: 0, Train Acc: 0.2500, Test Acc: 0.1667
Epoch: 0, Train Acc: 0.2583, Test Acc: 0.2000
Epoch: 0, Train Acc: 0.2583, Test Acc: 0.2000
Epoch: 10, Train Acc: 0.3667, Test Acc: 0.2333
Epoch: 10, Train Acc: 0.3667, Test Acc: 0.2333
Epoch: 10, Train Acc: 0.3667, Test Acc: 0.2333
Epoch: 10, Train Acc: 0.3667, Test Acc: 0.2333
Epoch: 10, Train Acc: 0.3583, Test Acc: 0.2333
Epoch: 10, Train Acc: 0.3583, Test Acc: 0.2333
Epoch: 10, Train Acc: 0.3583, Test Acc: 0.2333
Epoch: 10, Train Acc: 0.3583, Test Acc: 0.2333
Epoch: 20, Train Acc: 0.4417, Test Acc: 0.3333
Epoch: 20, Train Acc: 0.4667, Test Acc: 0.3667
Epoch: 20, Train Acc: 0.4500, Test Acc: 0.3667
Epoch: 20, Train Acc: 0.4667, Test Acc: 0.3667
Epoch: 20, Train Acc: 0.4000, Test Acc: 0.2333
Epoch: 20, Train Acc:

In [18]:
jax.devices()

[CpuDevice(id=0)]

In [19]:
x = jnp.ones([1000, 1000])
y = jnp.dot(x, x)

In [20]:
y.device

CpuDevice(id=0)